# Run the Domain-Specific RAG Chatbot in Google Colab

**Before you start**
1. Add a Colab Secret named `GROQ_API_KEY` (key icon on the left) and switch on *Notebook access*.
2. Runtime > Change runtime type > **T4 GPU** (faster embeddings, optional).
3. Upload `domain_rag_chatbot.zip` using the folder icon on the left (drag it into `/content`).
4. Run the cells below in order, one at a time.


## 1. Unzip and install

In [ ]:
!unzip -q -o /content/domain_rag_chatbot.zip -d /content
%cd /content/domain_rag_chatbot
!pip install -q -r requirements.txt

## 2. Set the Groq key

In [ ]:
import os
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
print('Key loaded:', bool(os.environ['GROQ_API_KEY']))

## 3. Offline unit tests (optional, no key needed)

In [ ]:
!python -m pytest -q tests

## 4. Evaluation sheet (retrieval, answers, refusals, response time)
Results are saved to `tests/evaluation_results.csv`.

In [ ]:
!python evaluate.py

## 5. Ask questions here (simple chat in the notebook)

In [ ]:
from rag_pipeline import RAGPipeline
import document_loader
pages, report = document_loader.load_folder()
pipe = RAGPipeline()
print('Chunks:', pipe.build(pages))
while True:
    q = input('Ask a question (or exit): ').strip()
    if q.lower() in ('exit', 'quit', ''):
        break
    r = pipe.ask(q)
    print('A:', r['answer'])
    if r['sources']:
        print('Sources:', '; '.join(f"{s['source']} (page {s['page']})" for s in r['sources']))
    print(f"[{r['seconds']} s]\n")

## 6. Streamlit app (optional live demo)
Run the cell, wait for the line `your url is: https://...loca.lt`, open it, and enter the password printed above it (your Colab public IP).

In [ ]:
!npm install -q -g localtunnel
import subprocess, time, urllib.request
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip()
print('Tunnel password (public IP):', ip)
subprocess.Popen(['streamlit','run','app.py','--server.port','8501','--server.headless','true'])
time.sleep(8)
!npx localtunnel --port 8501